The program should run error-free end-to-end. It fine-tunes DeepSeek R1 Distill Qwen 1.5B for a logical reasoning task: If A > B and B > C, then who is the largest?

### Import and Setup

In [1]:
import random
import json
import string
from pathlib import Path

# Fixed seed for reproducibility (required by the assignment)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Dataset parameters
DATASET_SIZE = 1000
OUTPUT_FILE = "logical_reasoning_dataset.jsonl"

# Letters we will use for variables (A, B, C, ...)
LETTERS = list(string.ascii_uppercase)

print(f"{LETTERS[:10]}")

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']


### Generate the Logical Reasoning Dataset

In [2]:
def generate_example():
    # randomly choose three different letters
    vars = random.sample(LETTERS, 3)
    A, B, C = vars[0], vars[1], vars[2]

    question = f"If {A} > {B} and {B} > {C}, who is the largest?"

    reasoning = (
        f"From {A} > {B}, {A} is larger than {B}. "
        f"From {B} > {C}, {B} is larger than {C}. "
        f"Therefore {A} > {B} > {C}. "
        f"So the largest is {A}."
    )

    answer = A

    return {
        "question": question,
        "reasoning": reasoning,
        "answer": answer
    }


dataset = []

for _ in range(DATASET_SIZE):
    dataset.append(generate_example())


# save dataset to JSONL file
with open(OUTPUT_FILE, "w") as f:
    for item in dataset:
        f.write(json.dumps(item) + "\n")


In [3]:
# Dataset preview
preview_samples = 5

with open(OUTPUT_FILE, "r") as f:
    lines = f.readlines()

print(f"Total examples in dataset: {len(lines)}\n")

for i in range(preview_samples):
    example = json.loads(lines[i])

    print(f"Example {i+1}")
    print("Question:", example["question"])
    print("Reasoning:", example["reasoning"])
    print("Answer:", example["answer"])
    print("\n")

Total examples in dataset: 1000

Example 1
Question: If U > D and D > A, who is the largest?
Reasoning: From U > D, U is larger than D. From D > A, D is larger than A. Therefore U > D > A. So the largest is U.
Answer: U


Example 2
Question: If X > I and I > H, who is the largest?
Reasoning: From X > I, X is larger than I. From I > H, I is larger than H. Therefore X > I > H. So the largest is X.
Answer: X


Example 3
Question: If H > E and E > X, who is the largest?
Reasoning: From H > E, H is larger than E. From E > X, E is larger than X. Therefore H > E > X. So the largest is H.
Answer: H


Example 4
Question: If D > V and V > X, who is the largest?
Reasoning: From D > V, D is larger than V. From V > X, V is larger than X. Therefore D > V > X. So the largest is D.
Answer: D


Example 5
Question: If R > C and C > S, who is the largest?
Reasoning: From R > C, R is larger than C. From C > S, C is larger than S. Therefore R > C > S. So the largest is R.
Answer: R




In [4]:
# Format the dataset for training

FORMATTED_OUTPUT_FILE = "logical_reasoning_training.jsonl"

formatted_data = []

with open(OUTPUT_FILE, "r") as f:
    for line in f:
        example = json.loads(line)

        training_text = (
            f"Question: {example['question']}\n"
            f"Think: {example['reasoning']}\n"
            f"Answer: {example['answer']}"
        )

        formatted_data.append({"text": training_text})


# save formatted dataset
with open(FORMATTED_OUTPUT_FILE, "w") as f:
    for item in formatted_data:
        f.write(json.dumps(item) + "\n")

print(f"Total training examples: {len(formatted_data)}")

# preview first example
print("\nExample formatted entry:\n")
print(formatted_data[0]["text"])

Total training examples: 1000

Example formatted entry:

Question: If U > D and D > A, who is the largest?
Think: From U > D, U is larger than D. From D > A, D is larger than A. Therefore U > D > A. So the largest is U.
Answer: U


### Fine-tuning deepseek-r1:1.5b

Prepare training data for Ollama

In [5]:
%pip install transformers datasets peft accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 14.8 MB/s eta 0:00:00


In [6]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

dataset = load_dataset("json", data_files=FORMATTED_OUTPUT_FILE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Model and dataset loaded.


In [7]:
# Apply LoRA

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,778,177,536 || trainable%: 0.0613


In [10]:
# Train Model
def formatting_func(example):
    return example["text"]

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./deepseek_reasoning_model",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    logging_steps=10,
    save_strategy="epoch",
    learning_rate=2e-4,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
    args=training_args,
    formatting_func=formatting_func,
)

trainer.train()

Applying formatting function to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151646, 'pad_token_id': 151643}.


Step,Training Loss
10,1.575116
20,1.165825
30,0.712855
40,0.357923
50,0.254436
60,0.233450
70,0.219433
80,0.192884
90,0.178398
100,0.179329


TrainOutput(global_step=1000, training_loss=0.20463285744190216, metrics={'train_runtime': 439.9096, 'train_samples_per_second': 4.546, 'train_steps_per_second': 2.273, 'total_flos': 1056645826560000.0, 'train_loss': 0.20463285744190216})

In [11]:
trainer.model.save_pretrained("deepseek_reasoning_lora")
tokenizer.save_pretrained("deepseek_reasoning_lora")

('deepseek_reasoning_lora/tokenizer_config.json',
 'deepseek_reasoning_lora/chat_template.jinja',
 'deepseek_reasoning_lora/tokenizer.json')

### Evaluating the fine-tuning model and compare it to the base model

In [26]:

# Compare Base Model vs Fine-Tuned Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

#  Load Models
BASE_MODEL = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
LORA_PATH = "deepseek_reasoning_lora"

# Tokenizer (same for both models)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load base model (original)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
base_model.eval()

# Load fine-tuned LoRA model
ft_model = PeftModel.from_pretrained(base_model, LORA_PATH)
ft_model.eval()

# Generate Random Names Questions-
names_list = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Henry", "Ivy", "Jack"]

num_questions = 10
test_questions = []

for _ in range(num_questions):
    chosen = random.sample(names_list, 3)
    question = f"If {chosen[0]} < {chosen[1]} and {chosen[1]} < {chosen[2]}, who is the largest?"
    test_questions.append(question)

# Inference Function
def ask_model(model, question, max_tokens=100, temperature=0.2):
    prompt = f"Question: {question}\nThink:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_tokens, temperature=temperature)
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response

# Compare Answers
for i, q in enumerate(test_questions):
    print(f"\nQuestion {i+1}: {q}\n")

    base_answer = ask_model(base_model, q)
    ft_answer = ask_model(ft_model, q)

    print("Original Base Model Answer:")
    print(base_answer)
    print("\nFine-Tuned Model Answer:")
    print(ft_answer)
    print("")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Question 1: If Eve < Henry and Henry < Diana, who is the largest?



Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Original Base Model Answer:
Question: If Eve < Henry and Henry < Diana, who is the largest?
Think: From Eve < Henry, Henry is larger than Eve. From Henry < Diana, Henry is smaller than Diana. Therefore Diana > Henry > Eve. So the largest is Diana.
Answer: Diana

Fine-Tuned Model Answer:
Question: If Eve < Henry and Henry < Diana, who is the largest?
Think: From Eve < Henry, Henry is larger than Eve. From Henry < Diana, Henry is smaller than Diana. Therefore Diana > Henry > Eve. So the largest is Diana.
Answer: Diana


Question 2: If Ivy < Charlie and Charlie < Henry, who is the largest?



...

Original Base Model Answer:
Question: If Frank < Bob and Bob < Diana, who is the largest?
Think: From Frank < Bob, Bob is larger than Frank. From Bob < Diana, Bob is smaller than Diana. Therefore Diana > Bob > Frank. So the largest is Diana.
Answer: Diana

Fine-Tuned Model Answer:
Question: If Frank < Bob and Bob < Diana, who is the largest?
Think: From Frank < Bob, Bob is larger than Frank. From Bob < Diana, Bob is smaller than Diana. Therefore Diana > Bob > Frank. So the largest is Diana.
Answer: Diana


Question 9: If Diana < Jack and Jack < Henry, who is the largest?



Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Original Base Model Answer:
Question: If Diana < Jack and Jack < Henry, who is the largest?
Think: From Diana < Jack, Jack is larger than Diana. From Jack < Henry, Henry is larger than Jack. Therefore Henry > Jack > Diana. So the largest is Henry.
Answer: Henry

Fine-Tuned Model Answer:
Question: If Diana < Jack and Jack < Henry, who is the largest?
Think: From Diana < Jack, Jack is larger than Diana. From Jack < Henry, Henry is larger than Jack. Therefore Henry > Jack > Diana. So the largest is Henry.
Answer: Henry


Question 10: If Charlie < Grace and Grace < Jack, who is the largest?



Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Original Base Model Answer:
Question: If Charlie < Grace and Grace < Jack, who is the largest?
Think: From Charlie < Grace, Grace is larger than Charlie. From Grace < Jack, Grace is smaller than Jack. Therefore Grace is the largest.
Answer: Grace

Fine-Tuned Model Answer:
Question: If Charlie < Grace and Grace < Jack, who is the largest?
Think: From Charlie < Grace, Grace is larger than Charlie. From Grace < Jack, Grace is smaller than Jack. Therefore Grace is the largest.
Answer: Grace

